# 00 — Setup: Schemas, Volumes, and Metadata Tables
This notebook creates all Unity Catalog objects and populates the parent metadata table.
Run this **once** before the pipeline.

In [0]:
# Widget parameters
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_name", "raw_zone", "Schema Name")
dbutils.widgets.text("base_path", "/Volumes/workspace/raw_zone/chinook", "Base Path")
dbutils.widgets.text("table_name", "all", "Table Name")
dbutils.widgets.text("run_id", "", "Run ID (auto-generated if blank)")

catalog = dbutils.widgets.get("catalog_name")
run_id_param = dbutils.widgets.get("run_id")

## 1. Create Schemas

In [0]:
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.raw_zone")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.metadata")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.quarantine")

print("All schemas created successfully.")

All schemas created successfully.


## 2. Create Raw Volume

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.raw_zone.chinook")
print(f"Volume created: {catalog}.raw_zone.chinook")

Volume created: workspace.raw_zone.chinook


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.metadata.pipeline_parent_metadata")

DataFrame[]

## 3. Create Parent Metadata Table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.metadata.pipeline_parent_metadata (
    table_id          INT,
    source_catalog    STRING,
    source_schema     STRING,
    source_table      STRING,
    table_name        STRING,
    file_name         STRING,
    primary_key_col   STRING,
    active_flag       STRING,
    load_sequence     INT,
    created_date      TIMESTAMP,
    modified_date     TIMESTAMP
)
USING DELTA
""")

print("Parent metadata table created.")

Parent metadata table created.


## 4. Create Child Execution Metrics Table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.metadata.pipeline_child_metrics (
    run_id              STRING,
    table_name          STRING,
    stage_name          STRING,
    execution_time      TIMESTAMP,
    status              STRING,
    source_row_count    BIGINT,
    target_row_count    BIGINT,
    records_passed_dq   BIGINT,
    records_failed_dq   BIGINT,
    file_location       STRING,
    error_message       STRING,
    start_time          TIMESTAMP,
    end_time            TIMESTAMP,
    duration_seconds    DOUBLE,
    created_date        TIMESTAMP
)
USING DELTA
""")

print("Child execution metrics table created.")

Child execution metrics table created.


## 5. Create DQ Log Table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.metadata.dq_execution_log (
    run_id              STRING,
    table_name          STRING,
    rule_name           STRING,
    rule_description    STRING,
    total_processed     BIGINT,
    passed_count        BIGINT,
    failed_count        BIGINT,
    execution_time      TIMESTAMP
)
USING DELTA
""")

print("DQ execution log table created.")

DQ execution log table created.


## 6. Populate Parent Metadata
One row per Chinook source table. Source references use the federation catalog `chinook_azure_sql_catalog`.

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Clear existing data for re-runs
spark.sql(f"TRUNCATE TABLE {catalog}.metadata.pipeline_parent_metadata")

# Define all Chinook tables — federation reference: chinook_azure_sql_catalog.chinook.<table>
chinook_tables = [
    (1,  "chinook_azure_sql_catalog", "chinook", "album",         "album",          "album.parquet",         "AlbumId",       "Y", 1),
    (2,  "chinook_azure_sql_catalog", "chinook", "artist",        "artist",         "artist.parquet",        "ArtistId",      "Y", 2),
    (3,  "chinook_azure_sql_catalog", "chinook", "customer",      "customer",       "customer.parquet",      "CustomerId",    "Y", 3),
    (4,  "chinook_azure_sql_catalog", "chinook", "employee",      "employee",       "employee.parquet",      "EmployeeId",    "Y", 4),
    (5,  "chinook_azure_sql_catalog", "chinook", "genre",         "genre",          "genre.parquet",         "GenreId",       "Y", 5),
    (6,  "chinook_azure_sql_catalog", "chinook", "invoice",       "invoice",        "invoice.parquet",       "InvoiceId",     "Y", 6),
    (7,  "chinook_azure_sql_catalog", "chinook", "invoiceline",   "invoiceline",    "invoiceline.parquet",   "InvoiceLineId", "Y", 7),
    (8,  "chinook_azure_sql_catalog", "chinook", "mediatype",     "mediatype",      "mediatype.parquet",     "MediaTypeId",   "Y", 8),
    (9,  "chinook_azure_sql_catalog", "chinook", "playlist",      "playlist",       "playlist.parquet",      "PlaylistId",    "Y", 9),
    (10, "chinook_azure_sql_catalog", "chinook", "playlisttrack", "playlisttrack",  "playlisttrack.parquet", "PlaylistId",    "Y", 10),
    (11, "chinook_azure_sql_catalog", "chinook", "track",         "track",          "track.parquet",         "TrackId",       "Y", 11),
]

columns = ["table_id", "source_catalog", "source_schema", "source_table", "table_name", "file_name", "primary_key_col", "active_flag", "load_sequence"]
df_meta = spark.createDataFrame(chinook_tables, columns)
df_meta = df_meta.withColumn("created_date", current_timestamp()).withColumn("modified_date", current_timestamp())

df_meta.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.metadata.pipeline_parent_metadata")

display(spark.sql(f"SELECT * FROM {catalog}.metadata.pipeline_parent_metadata ORDER BY load_sequence"))

table_id,source_catalog,source_schema,source_table,table_name,file_name,primary_key_col,active_flag,load_sequence,created_date,modified_date
1,chinook_azure_sql_catalog,chinook,album,album,album.parquet,AlbumId,Y,1,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z
2,chinook_azure_sql_catalog,chinook,artist,artist,artist.parquet,ArtistId,Y,2,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z
3,chinook_azure_sql_catalog,chinook,customer,customer,customer.parquet,CustomerId,Y,3,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z
4,chinook_azure_sql_catalog,chinook,employee,employee,employee.parquet,EmployeeId,Y,4,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z
5,chinook_azure_sql_catalog,chinook,genre,genre,genre.parquet,GenreId,Y,5,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z
6,chinook_azure_sql_catalog,chinook,invoice,invoice,invoice.parquet,InvoiceId,Y,6,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z
7,chinook_azure_sql_catalog,chinook,invoiceline,invoiceline,invoiceline.parquet,InvoiceLineId,Y,7,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z
8,chinook_azure_sql_catalog,chinook,mediatype,mediatype,mediatype.parquet,MediaTypeId,Y,8,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z
9,chinook_azure_sql_catalog,chinook,playlist,playlist,playlist.parquet,PlaylistId,Y,9,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z
10,chinook_azure_sql_catalog,chinook,playlisttrack,playlisttrack,playlisttrack.parquet,PlaylistId,Y,10,2026-03-28T05:11:05.844Z,2026-03-28T05:11:05.844Z


## 7. Verify Setup

In [0]:
print(f"Schemas in {catalog}:")
display(spark.sql(f"SHOW SCHEMAS IN {catalog}"))

print(f"\nVolumes in {catalog}.raw_zone:")
display(spark.sql(f"SHOW VOLUMES IN {catalog}.raw_zone"))

print(f"\nParent metadata rows:")
display(spark.sql(f"SELECT table_name, source_catalog, source_schema, source_table, active_flag, load_sequence FROM {catalog}.metadata.pipeline_parent_metadata ORDER BY load_sequence"))

Schemas in workspace:


databaseName
bronze
default
demo_snowday
dmo_snowday
gold
information_schema
metadata
quarantine
raw_zone
silver



Volumes in workspace.raw_zone:


database,volume_name
raw_zone,chinook



Parent metadata rows:


table_name,source_catalog,source_schema,source_table,active_flag,load_sequence
album,chinook_azure_sql_catalog,chinook,album,Y,1
artist,chinook_azure_sql_catalog,chinook,artist,Y,2
customer,chinook_azure_sql_catalog,chinook,customer,Y,3
employee,chinook_azure_sql_catalog,chinook,employee,Y,4
genre,chinook_azure_sql_catalog,chinook,genre,Y,5
invoice,chinook_azure_sql_catalog,chinook,invoice,Y,6
invoiceline,chinook_azure_sql_catalog,chinook,invoiceline,Y,7
mediatype,chinook_azure_sql_catalog,chinook,mediatype,Y,8
playlist,chinook_azure_sql_catalog,chinook,playlist,Y,9
playlisttrack,chinook_azure_sql_catalog,chinook,playlisttrack,Y,10
